In [ ]:
!git clone https://github.com/RodrickKamsiyonna/le-wm_kaggle.git
%cd le-wm_kaggle

In [ ]:
#rm -r /kaggle/working/le-wm_kaggle

In [ ]:
#cd /kaggle/working

In [ ]:
!apt-get install -y zstd
!mkdir /kaggle/data
!pip install "huggingface-hub>=0.34.0,<1.0" "datasets<3.0" "transformers>=4.45.0" --break-system-packages

In [ ]:
!pip install hydra-core  lightning
!pip install -U "stable-worldmodel[all]==0.0.6" "stable-pretraining==0.1.6"
import os

os.environ["HYDRA_FULL_ERROR"]="1"

In [ ]:
import os

os.makedirs("/kaggle/working/stablewm", exist_ok=True)
os.environ["STABLEWM_HOME"] = "/kaggle/working/stablewm"

print("STABLEWM_HOME set to:", os.environ["STABLEWM_HOME"])

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token

url = "https://huggingface.co/datasets/quentinll/lewm-tworooms/resolve/main/tworoom.tar.zst"
output = "/kaggle/working/data/tworoom.tar.zst"

os.makedirs("/kaggle/working/data", exist_ok=True)

# -C - enables resume if interrupted
!wget -c --header="Authorization: Bearer {hf_token}" "{url}" -O "{output}"

In [ ]:
mv /kaggle/working/data/tworoom.tar.zst /kaggle/data/tworoom.tar.zst

In [ ]:
!tar --zstd -xvf /kaggle/data/tworoom.tar.zst \
    -C /kaggle/working/stablewm/

# verify .h5 files are present
!find /kaggle/working/stablewm/ -name "*.h5"

In [ ]:
import wandb
wandb.login(key="6d6711a109e2c810d7cfd505ad774175c0ffb2b2")

In [ ]:
filepath = "/kaggle/working/le-wm_kaggle/config/train/lewm.yaml"

with open(filepath) as f:
    content = f.read()

# Fix wandb entity and reduce batch size while we're here
content = content.replace("entity: lewm", "entity: rodrickkamsi2-afe-babalola-university")
content = content.replace("project: lewm", "project: lewm-tworooms-phenux")


with open(filepath, "w") as f:
    f.write(content)

print("Done - verify:")
for line in content.split("\n"):
    if "entity" in line:
        print(" ", line.strip())

In [ ]:
filepath = "/kaggle/working/le-wm_kaggle/config/eval/tworoom.yaml"

with open(filepath) as f:
    content = f.read()

# Fix wandb entity and reduce batch size while we're here
content = content.replace("n_iter: 200", "n_iter: 10")

with open(filepath, "w") as f:
    f.write(content)

print("Done - verify:")
for line in content.split("\n"):
    if "entity" in line:
        print(" ", line.strip())

In [ ]:
# Create the expected directory structure
!mkdir -p /kaggle/working/stablewm/datasets

# Link your file into that directory
!ln -sf /kaggle/working/stablewm/tworoom.h5 /kaggle/working/stablewm/datasets/tworoom.h5

In [ ]:
#!python eval.py --config-name=tworoom.yaml policy=/kaggle/working/lewm_run/lewm_epoch_5

In [ ]:
#!python diagram.py --config-name=tworoom.yaml policy=/kaggle/working/lewm_run/lewm_epoch_5

In [ ]:
!python train.py data=tworoom